# Neural Network model for automatic segmentation of ventilation regimes during geneal anesthesia recovery phase

In [ ]:
import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt

## Setup config for Colab

In [ ]:
# Optional: Mount Google Drive so weights & dataset persist across Colab sessions
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/anesthesia_project"
else:
    BASE_DIR = "/content"

DATASET_DIR = os.path.join(BASE_DIR, "anesthesia_synth_dataset")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "unet_baseline_artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {DEVICE}")

## Dataset loader

_Why division into 60 s windows ?_

In [ ]:
class AnesthesiaWindowDataset(Dataset):
    """
    Slices full 500s recordings (3, 64000) into fixed-length windows (e.g., 60s = 7680 samples)
    and normalizes channels using strictly training-set mean & std.
    """
    def __init__(self, X, y, mean, std, window_samples=7680, stride_samples=3840):
        self.X = X
        self.y = y
        self.mean = mean[:, None]  # (3, 1)
        self.std = std[:, None]    # (3, 1)
        self.window_samples = window_samples

        self.indices = []
        n_cases, _, total_len = X.shape
        for c in range(n_cases):
            for start in range(0, total_len - window_samples + 1, stride_samples):
                self.indices.append((c, start))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        case_idx, start = self.indices[idx]
        end = start + self.window_samples
        x_win = (self.X[case_idx, :, start:end] - self.mean) / (self.std + 1e-6)
        y_win = self.y[case_idx, start:end]
        return torch.from_numpy(x_win).float(), torch.from_numpy(y_win).long()

## UNET standard model as reference

In [ ]:
class DoubleConv1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=7, dropout=0.1):
        super().__init__()
        pad = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=pad, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size=kernel_size, padding=pad, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class UNet1D(nn.Module):
    """
    4-level 1D U-Net for dense temporal segmentation of physiological waveforms.
    Input:  (B, 3, L) -> [CO2, AWP, Cdyn]
    Output: (B, num_classes, L)
    """
    def __init__(self, in_channels=3, num_classes=5, base_filters=32, dropout=0.1):
        super().__init__()
        f = base_filters

        # Encoder
        self.enc1 = DoubleConv1D(in_channels, f, kernel_size=7, dropout=dropout)
        self.pool1 = nn.MaxPool1d(4)  # /4

        self.enc2 = DoubleConv1D(f, f * 2, kernel_size=7, dropout=dropout)
        self.pool2 = nn.MaxPool1d(4)  # /16

        self.enc3 = DoubleConv1D(f * 2, f * 4, kernel_size=5, dropout=dropout)
        self.pool3 = nn.MaxPool1d(4)  # /64

        self.enc4 = DoubleConv1D(f * 4, f * 8, kernel_size=5, dropout=dropout)
        self.pool4 = nn.MaxPool1d(4)  # /256

        # Bottleneck (1 sample here = 2 seconds at 128 Hz)
        self.bottleneck = DoubleConv1D(f * 8, f * 16, kernel_size=5, dropout=dropout)

        # Decoder
        self.up4 = nn.ConvTranspose1d(f * 16, f * 8, kernel_size=4, stride=4)
        self.dec4 = DoubleConv1D(f * 16, f * 8, kernel_size=5, dropout=dropout)

        self.up3 = nn.ConvTranspose1d(f * 8, f * 4, kernel_size=4, stride=4)
        self.dec3 = DoubleConv1D(f * 8, f * 4, kernel_size=5, dropout=dropout)

        self.up2 = nn.ConvTranspose1d(f * 4, f * 2, kernel_size=4, stride=4)
        self.dec2 = DoubleConv1D(f * 4, f * 2, kernel_size=7, dropout=dropout)

        self.up1 = nn.ConvTranspose1d(f * 2, f, kernel_size=4, stride=4)
        self.dec1 = DoubleConv1D(f * 2, f, kernel_size=7, dropout=dropout)

        self.head = nn.Conv1d(f, num_classes, kernel_size=1)

    def _pad_and_cat(self, up_x, skip_x):
        diff = skip_x.size(-1) - up_x.size(-1)
        if diff > 0:
            up_x = F.pad(up_x, (0, diff))
        elif diff < 0:
            up_x = up_x[..., :skip_x.size(-1)]
        return torch.cat([skip_x, up_x], dim=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        d4 = self.dec4(self._pad_and_cat(self.up4(b), e4))
        d3 = self.dec3(self._pad_and_cat(self.up3(d4), e3))
        d2 = self.dec2(self._pad_and_cat(self.up2(d3), e2))
        d1 = self.dec1(self._pad_and_cat(self.up1(d2), e1))

        return self.head(d1)

## Inference code for test evaluation

In [ ]:
@torch.no_grad()
def predict_full_cases(model, X, mean, std, window_samples=7680, stride_samples=1920, batch_size=32):
    """
    Predicts full-length recordings (N_cases, total_len) by averaging overlapping
    window softmax probabilities so boundaries are smooth and artifact-free.
    """
    model.eval()
    n_cases, _, total_len = X.shape
    num_classes = model.head.out_channels

    prob_accum = np.zeros((n_cases, num_classes, total_len), dtype=np.float32)
    weight_accum = np.zeros((1, 1, total_len), dtype=np.float32)

    # Create a Hann-like center weight so window edges contribute less than centers
    win_weight = np.hanning(window_samples).astype(np.float32) + 0.1
    win_weight = win_weight[None, None, :]

    starts = list(range(0, total_len - window_samples + 1, stride_samples))
    if starts[-1] + window_samples < total_len:
        starts.append(total_len - window_samples)

    mean_b = mean[None, :, None]
    std_b = std[None, :, None]

    for start in starts:
        end = start + window_samples
        batch_np = (X[:, :, start:end] - mean_b) / (std_b + 1e-6)

        probs_list = []
        for b_idx in range(0, n_cases, batch_size):
            xb = torch.from_numpy(batch_np[b_idx:b_idx + batch_size]).float().to(DEVICE)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            probs_list.append(probs)

        all_probs = np.concatenate(probs_list, axis=0)
        prob_accum[:, :, start:end] += all_probs * win_weight
        weight_accum[:, :, start:end] += win_weight

    avg_probs = prob_accum / np.maximum(weight_accum, 1e-6)
    preds = np.argmax(avg_probs, axis=1)
    return preds, avg_probs

## Training and evaluation

In [ ]:
def train_and_evaluate(
    epochs=20,
    batch_size=32,
    lr=1e-3,
    window_sec=60,
    train_stride_sec=30,
    base_filters=32,
):
    # Load metadata & data
    with open(os.path.join(DATASET_DIR, "metadata.json"), "r") as f:
        metadata = json.load(f)

    fs = metadata["fs"]
    channels = metadata["channels"]
    regime_names = {int(k): v for k, v in metadata["regimes"].items()}
    num_classes = len(regime_names)

    mean = np.array([metadata["train_normalization"]["mean"][c] for c in channels], dtype=np.float32)
    std = np.array([metadata["train_normalization"]["std"][c] for c in channels], dtype=np.float32)

    print("Loading dataset splits...")
    train_npz = np.load(os.path.join(DATASET_DIR, "train.npz"))
    val_npz = np.load(os.path.join(DATASET_DIR, "val.npz"))
    test_npz = np.load(os.path.join(DATASET_DIR, "test.npz"))

    X_train, y_train = train_npz["X"], train_npz["y"]
    X_val, y_val = val_npz["X"], val_npz["y"]
    X_test, y_test = test_npz["X"], test_npz["y"]

    win_samples = int(window_sec * fs)
    stride_samples = int(train_stride_sec * fs)

    train_ds = AnesthesiaWindowDataset(X_train, y_train, mean, std, win_samples, stride_samples)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

    # Inverse-frequency class weights to prevent majority Controlled class from dominating
    _, class_counts = np.unique(y_train, return_counts=True)
    class_weights = (class_counts.sum() / (num_classes * class_counts)) ** 0.5
    class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
    print(f"Class weights: {dict(zip(regime_names.values(), np.round(class_weights, 3)))}")

    model = UNet1D(in_channels=3, num_classes=num_classes, base_filters=base_filters, dropout=0.1).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(weight=class_weights_t)

    best_val_f1 = -1.0
    history = []
    weights_path = os.path.join(ARTIFACTS_DIR, "best_unet1d.pt")

    print(f"\nStarting training for {epochs} epochs ({len(train_ds)} training windows)...")
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)

        scheduler.step()
        train_loss = running_loss / len(train_ds)

        # Full-case validation evaluation
        val_preds, _ = predict_full_cases(model, X_val, mean, std, window_samples=win_samples, stride_samples=win_samples // 2)
        val_macro_f1 = f1_score(y_val.ravel(), val_preds.ravel(), average="macro")
        val_acc = float((val_preds == y_val).mean())

        elapsed = time.time() - t0
        print(
            f"Epoch {epoch:02d}/{epochs:02d} [{elapsed:.1f}s] | "
            f"Train Loss: {train_loss:.4f} | Val Acc: {val_acc*100:.2f}% | Val Macro-F1: {val_macro_f1:.4f}"
        )
        history.append({"epoch": epoch, "train_loss": train_loss, "val_acc": val_acc, "val_macro_f1": val_macro_f1})

        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "model_config": {
                        "in_channels": 3,
                        "num_classes": num_classes,
                        "base_filters": base_filters,
                        "window_samples": win_samples,
                    },
                    "normalization": {"mean": mean.tolist(), "std": std.tolist()},
                    "regime_names": regime_names,
                    "fs": fs,
                    "best_val_macro_f1": best_val_f1,
                },
                weights_path,
            )

    print(f"\nBest model saved to {weights_path} (Val Macro-F1: {best_val_f1:.4f})")

    # ==========================================================================
    # 6. FINAL TEST SET EVALUATION
    # ==========================================================================
    print("\nRunning full-case evaluation on TEST set...")
    ckpt = torch.load(weights_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])

    test_preds, _ = predict_full_cases(
        model, X_test, mean, std, window_samples=win_samples, stride_samples=win_samples // 4
    )

    y_true_flat = y_test.ravel()
    y_pred_flat = test_preds.ravel()

    target_names = [regime_names[i] for i in range(num_classes)]
    report_dict = classification_report(y_true_flat, y_pred_flat, target_names=target_names, output_dict=True, digits=4)
    report_text = classification_report(y_true_flat, y_pred_flat, target_names=target_names, digits=4)
    cm = confusion_matrix(y_true_flat, y_pred_flat)

    print("\n--- TEST SET CLASSIFICATION REPORT ---")
    print(report_text)

    # Save JSON evaluation report
    eval_summary = {
        "best_val_macro_f1": best_val_f1,
        "test_overall_accuracy": float((y_true_flat == y_pred_flat).mean()),
        "test_macro_f1": float(report_dict["macro avg"]["f1-score"]),
        "per_class_metrics": report_dict,
        "confusion_matrix": cm.tolist(),
        "training_history": history,
    }
    eval_json_path = os.path.join(ARTIFACTS_DIR, "eval_results.json")
    with open(eval_json_path, "w") as f:
        json.dump(eval_summary, f, indent=2)

    # Plot & save normalized confusion matrix
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(num_classes))
    ax.set_yticks(range(num_classes))
    ax.set_xticklabels(target_names, rotation=30, ha="right")
    ax.set_yticklabels(target_names)
    ax.set_xlabel("Predicted Regime")
    ax.set_ylabel("True Regime")
    ax.set_title("1D U-Net Test Set Confusion Matrix (Row-Normalized)")

    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(j, i, f"{cm_norm[i, j]*100:.1f}%", ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=9)

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    cm_png_path = os.path.join(ARTIFACTS_DIR, "confusion_matrix.png")
    plt.savefig(cm_png_path, dpi=150)
    plt.show()

    # Zip artifacts so you can download everything in one click from Colab
    zip_path = os.path.join(BASE_DIR, "unet_baseline_artifacts.zip")
    os.system(f"zip -r -q {zip_path} {ARTIFACTS_DIR}")
    print(f"\nAll weights & results zipped to: {zip_path}")
    print("In Colab, run: from google.colab import files; files.download('/content/unet_baseline_artifacts.zip')")

## Run

In [ ]:
train_and_evaluate(epochs=30, batch_size=32, lr=1e-3)